# Fase 3 - QSVM com Quantum Kernel e Roteamento por Regime

**Estratégia em duas partes:**
1. **QSVM Classificador** — detecta o regime epidemiológico (declínio / endemia / crescimento) usando o Quantum Kernel ZZ Feature Map (Havlíček et al., 2019, *Nature*).
2. **QSVM Regressor com roteamento** — encaminha cada semana ao SVR especializado no seu regime, explorando a heterogeneidade da dinâmica do dengue no DF.

**Hipótese:** um modelo que *primeiro identifica o regime* e *depois regride dentro dele* tem melhor WIS do que um único regressor global, especialmente na transição C1→C2 (início do surto).

In [11]:
try:
    import mlflow, mlflow.sklearn
    mlflow.set_tracking_uri("mlruns")
    _MLFLOW = False  # tracking desativado (entregável)
except ImportError:
    _MLFLOW = False
    print("[AVISO] mlflow nao instalado — execute: pip install mlflow")
import warnings; warnings.filterwarnings("ignore")
import os, json, sys, time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, os.path.join(REPO_ROOT, "src"))
from feature_engineering import construir_features, splits_validacao

CACHE = os.path.join(REPO_ROOT, "data", "dados_dengue_df_real.json")
with open(CACHE, encoding="utf-8") as f:
    dados_brutos = json.load(f)

dataset = construir_features(dados_brutos, n_lags=4)
splits  = splits_validacao(dataset)

NOMES = {0: ("C1", "Transmissão normal/crescente  (out/2023 – out/2024)"),
         1: ("C2", "Pico recorde 25.714 casos/sem  (jun/2024 – jun/2025)"),
         2: ("C3", "Pós-surto, Rt < 1              (out/2024 – jun/2025)")}

CENARIOS = {}
for idx, split in enumerate(splits[:3]):
    nome, desc = NOMES[idx]
    tr, te = split["treino"], split["teste"]
    CENARIOS[nome] = {
        "X_train": np.array(tr["X"]), "y_train": np.array(tr["y"]),
        "X_test":  np.array(te["X"]), "y_test":  np.array(te["y"]),
        "datas":   te["datas"], "nome": desc,
    }

print(f"Features ({len(dataset['feature_names'])}): {dataset['feature_names']}")
for nome, d in CENARIOS.items():
    print(f"{nome}: treino={len(d['X_train'])} | teste={len(d['X_test'])} | "
          f"target_max={max(d['y_test']):.0f}")

# ── utilitários compartilhados (utils_qml.py na raiz do projeto) ─────────────
import sys as _sys, os as _os
_sys.path.insert(0, _os.path.abspath(".."))
from utils_qml import (calcular_wis, metricas, salvar_padrao, plot_pred,
                        validar_json_saida, validar_pipeline,
                        testar_invariancia_quantica, testar_propriedades,
                        validar_golden)
testar_propriedades()
print("[OK] utils_qml importado")

Features (13): ['casos_est_lag1', 'casos_est_lag2', 'casos_est_lag3', 'casos_est_lag4', 'Rt_lag1', 'Rt_lag2', 'p_rt1_lag1', 'receptivo_lag1', 'transmissao_lag1', 'tempmed_lag1', 'umidmed_lag1', 'SE_sin', 'SE_cos']
C1: treino=36 | teste=143 | target_max=25714
C2: treino=88 | teste=91 | target_max=25714
C3: treino=125 | teste=54 | target_max=947
[HYPOTHESIS] Biblioteca nao instalada. Executando versao simplificada.
             Para instalar: pip install hypothesis
[PROP OK] 500 combinacoes aleatorias: WIS>=0, RMSE>=0, MAE>=0 em todos.
[OK] utils_qml importado


In [12]:
import pennylane as qml
from pennylane import numpy as pnp
from sklearn.svm import SVR, SVC
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, classification_report
print(f"PennyLane: {qml.__version__}")

# ── Configuração ──────────────────────────────────────────────────────────
CONFIG = {
    "n_qubits": 6,
    "n_layers_kernel": 2,   # profundidade do feature map quântico
    "seed": 42,
    "C_svm": 10.0,          # regularização do SVM
    "epsilon_svr": 0.1,     # margem do SVR
}
np.random.seed(CONFIG["seed"])

# Índices das features epidemiológicas-chave (usadas no kernel)
# Rt_lag1=4, Rt_lag2=5, p_rt1_lag1=6, receptivo_lag1=7, transmissao_lag1=8
FEAT_IDX = [0, 1, 2, 3, 4, 5, 6, 7]   # casos_est_lag1..4 + Rt_lag1/2 + p_rt1 + receptivo
N_FEAT_KERNEL = len(FEAT_IDX)
print(f"Features para o kernel quântico ({N_FEAT_KERNEL}): "
      f"{[dataset['feature_names'][i] for i in FEAT_IDX]}")

PennyLane: 0.40.0
Features para o kernel quântico (8): ['casos_est_lag1', 'casos_est_lag2', 'casos_est_lag3', 'casos_est_lag4', 'Rt_lag1', 'Rt_lag2', 'p_rt1_lag1', 'receptivo_lag1']


In [13]:
# ── Rótulos de regime epidemiológico ─────────────────────────────────────
# Baseado no Rt_lag1 (índice 4 no vetor de features)
RT_IDX = 4   # posição de Rt_lag1 nas features

def rotular_regime(X_raw):
    """
    0 = declínio      (Rt < 0.85)
    1 = endemia estável (0.85 ≤ Rt ≤ 1.20)
    2 = crescimento   (Rt > 1.20)
    Limites calibrados para o DF (2022-2025).
    """
    rt = X_raw[:, RT_IDX]
    labels = np.ones(len(rt), dtype=int)        # default: endemia
    labels[rt < 0.85]  = 0                      # declínio
    labels[rt > 1.20]  = 2                      # crescimento
    return labels

# Prepara dados de regime para cada cenário
for cen, d in CENARIOS.items():
    d["regime_train"] = rotular_regime(d["X_train"])
    d["regime_test"]  = rotular_regime(d["X_test"])
    contagem = {r: int(np.sum(d["regime_train"]==r)) for r in [0,1,2]}
    print(f"{cen}: regimes treino → {contagem} | "
          f"teste → {dict((r, int(np.sum(d['regime_test']==r))) for r in [0,1,2])}")

print("\nLegenda: 0=declínio | 1=endemia | 2=crescimento")

C1: regimes treino → {0: 30, 1: 6, 2: 0} | teste → {0: 124, 1: 6, 2: 13}
C2: regimes treino → {0: 79, 1: 9, 2: 0} | teste → {0: 75, 1: 3, 2: 13}
C3: regimes treino → {0: 103, 1: 11, 2: 11} | teste → {0: 51, 1: 1, 2: 2}

Legenda: 0=declínio | 1=endemia | 2=crescimento


In [14]:
try:
    _dev_name = "lightning.qubit"
    import pennylane_lightning  # noqa
except ImportError:
    _dev_name = "default.qubit"
    print("[AVISO] pennylane-lightning nao instalado, usando default.qubit (mais lento)")
# ── Quantum Kernel (ZZ Feature Map) ──────────────────────────────────────
n_q = CONFIG["n_qubits"]
dev_k = qml.device(_dev_name, wires=n_q)

@qml.qnode(dev_k)
def feature_map(x):
    """
    ZZ Feature Map: Hadamard + RZ(xᵢ) + CZ + RZ(xᵢ·xⱼ) por par de qubits.
    Inspirado em Havlíček et al. (2019), Nature.
    """
    x_pad = x[:n_q] if len(x) >= n_q else np.pad(x, (0, n_q - len(x)))
    for rep in range(CONFIG["n_layers_kernel"]):
        for i in range(n_q):
            qml.Hadamard(wires=i)
            qml.RZ(x_pad[i % len(x_pad)], wires=i)
        for i in range(n_q - 1):
            qml.CZ(wires=[i, i+1])
            fi = x_pad[i % len(x_pad)]
            fj = x_pad[(i+1) % len(x_pad)]
            qml.RZ((np.pi - fi) * (np.pi - fj), wires=i)
            qml.RZ((np.pi - fi) * (np.pi - fj), wires=i+1)
    return qml.state()

def quantum_kernel(x1, x2):
    """K(x1,x2) = |⟨φ(x1)|φ(x2)⟩|²"""
    s1 = feature_map(x1)
    s2 = feature_map(x2)
    return float(np.abs(np.dot(np.conj(s1), s2)) ** 2)

def build_kernel_matrix(X_a, X_b, verbose=False):
    """Constrói a matriz de kernel K[i,j] = K(X_a[i], X_b[j])."""
    n_a, n_b = len(X_a), len(X_b)
    K = np.zeros((n_a, n_b))
    total = n_a * n_b
    for i in range(n_a):
        for j in range(n_b):
            K[i, j] = quantum_kernel(X_a[i], X_b[j])
        if verbose and (i+1) % 5 == 0:
            print(f"  Kernel: {i+1}/{n_a} linhas ({(i+1)*n_b}/{total} avaliações)")
    return K

# Visualização do circuito
x_viz = np.linspace(0, np.pi, n_q)
fig, _ = qml.draw_mpl(feature_map)(pnp.array(x_viz))
plt.title("Fase 6 — ZZ Feature Map (Quantum Kernel)")
plt.tight_layout(); plt.show()
print(f"[OK] Quantum kernel definido: {n_q} qubits | {CONFIG['n_layers_kernel']} repetições")

[OK] Quantum kernel definido: 6 qubits | 2 repetições


In [15]:
# ── Parte 1: Classificador de Regime (QSVM-C) ────────────────────────────
print("=" * 65)
print("  QSVM CLASSIFICADOR DE REGIME")
print("=" * 65)

RESULTADOS_CLF = {}
for cen, d in CENARIOS.items():
    print(f"\n  Cenário {cen}...")
    t0 = time.time()

    # Seleciona e normaliza features para o kernel
    sc = MinMaxScaler(feature_range=(0.01, np.pi))
    X_tr_k = sc.fit_transform(d["X_train"][:, FEAT_IDX])
    X_te_k = sc.transform(d["X_test"][:, FEAT_IDX])
    y_tr_r = d["regime_train"]
    y_te_r = d["regime_test"]

    print(f"    Construindo kernel treino ({len(X_tr_k)}×{len(X_tr_k)})...")
    K_train = build_kernel_matrix(X_tr_k, X_tr_k, verbose=True)

    print(f"    Construindo kernel teste ({len(X_te_k)}×{len(X_tr_k)})...")
    K_test  = build_kernel_matrix(X_te_k, X_tr_k)

    clf = SVC(kernel="precomputed", C=CONFIG["C_svm"], random_state=42)
    clf.fit(K_train, y_tr_r)
    y_pred_r = clf.predict(K_test)

    acc = accuracy_score(y_te_r, y_pred_r)
    print(f"    Acurácia de regime: {acc:.4f} | {time.time()-t0:.0f}s")
    print(classification_report(y_te_r, y_pred_r,
                                 target_names=["Declínio","Endemia","Crescimento"],
                                 zero_division=0))

    RESULTADOS_CLF[cen] = {
        "acc": acc, "y_true": y_te_r, "y_pred": y_pred_r,
        "K_train": K_train, "K_test": K_test,
        "X_tr_k": X_tr_k, "X_te_k": X_te_k,
        "scaler": sc, "clf": clf,
    }

print("[OK] Classificação de regime concluída")

  QSVM CLASSIFICADOR DE REGIME

  Cenário C1...
    Construindo kernel treino (36×36)...
  Kernel: 5/36 linhas (180/1296 avaliações)
  Kernel: 10/36 linhas (360/1296 avaliações)
  Kernel: 15/36 linhas (540/1296 avaliações)
  Kernel: 20/36 linhas (720/1296 avaliações)
  Kernel: 25/36 linhas (900/1296 avaliações)
  Kernel: 30/36 linhas (1080/1296 avaliações)
  Kernel: 35/36 linhas (1260/1296 avaliações)
    Construindo kernel teste (143×36)...
    Acurácia de regime: 0.8601 | 82s
              precision    recall  f1-score   support

    Declínio       0.87      0.99      0.92       124
     Endemia       0.00      0.00      0.00         6
 Crescimento       0.00      0.00      0.00        13

    accuracy                           0.86       143
   macro avg       0.29      0.33      0.31       143
weighted avg       0.75      0.86      0.80       143


  Cenário C2...
    Construindo kernel treino (88×88)...
  Kernel: 5/88 linhas (440/7744 avaliações)
  Kernel: 10/88 linhas (880/7744 a

In [16]:
# ── Parte 2: Regressão com Roteamento por Regime (QSVM-R) ────────────────
print("=" * 65)
print("  QSVM REGRESSÃO COM ROTEAMENTO POR REGIME")
print("=" * 65)

# Estratégia: treina um SVR por regime com kernel quântico
# O regime previsto (Parte 1) roteia a amostra para o SVR correspondente

N_BOOT = 5
np.random.seed(42)
RESULTADOS = {}

for cen, d in CENARIOS.items():
    print(f"\n  Cenário {cen}...")
    t0 = time.time()

    sc    = RESULTADOS_CLF[cen]["scaler"]
    clf_r = RESULTADOS_CLF[cen]["clf"]
    K_tr  = RESULTADOS_CLF[cen]["K_train"]
    K_te  = RESULTADOS_CLF[cen]["K_test"]
    X_tr_k = RESULTADOS_CLF[cen]["X_tr_k"]
    X_te_k = RESULTADOS_CLF[cen]["X_te_k"]

    y_tr   = d["y_train"]
    y_te   = d["y_test"]
    regimes_tr = d["regime_train"]
    regimes_te = RESULTADOS_CLF[cen]["y_pred"]   # usa regime previsto

    # Treina SVR separado por regime
    svrs = {}
    for r in [0, 1, 2]:
        mask = regimes_tr == r
        if mask.sum() < 3:
            print(f"    Regime {r}: poucos exemplos ({mask.sum()}), usa SVR global")
            svrs[r] = None
        else:
            svr = SVR(kernel="precomputed", C=CONFIG["C_svm"],
                      epsilon=CONFIG["epsilon_svr"])
            svr.fit(K_tr[np.ix_(mask, mask)], np.log1p(y_tr[mask]))
            svrs[r] = (svr, np.where(mask)[0])

    # Fallback: SVR global
    svr_global = SVR(kernel="precomputed", C=CONFIG["C_svm"],
                     epsilon=CONFIG["epsilon_svr"])
    svr_global.fit(K_tr, np.log1p(y_tr))

    # Bootstrap sobre os regimes de treino (perturbação de labels)
    preds_matrix = np.zeros((N_BOOT, len(y_te)))
    for b in range(N_BOOT):
        rng = np.random.RandomState(42 + b)
        # Perturba 10% dos labels de regime para estimar incerteza
        noise_mask = rng.rand(len(regimes_te)) < 0.10
        reg_boot = regimes_te.copy()
        reg_boot[noise_mask] = rng.randint(0, 3, noise_mask.sum())

        preds = np.zeros(len(y_te))
        for i, reg in enumerate(reg_boot):
            k_row = K_te[i]
            if svrs.get(reg) is not None:
                svr_r, tr_idx = svrs[reg]
                k_sub = K_te[i, tr_idx].reshape(1, -1)
                # fallback se kernel sub-matrix tiver shape errado
                try:
                    preds[i] = np.expm1(float(svr_r.predict(k_sub)))
                except Exception:
                    preds[i] = np.expm1(float(svr_global.predict(k_row.reshape(1,-1))))
            else:
                preds[i] = np.expm1(float(svr_global.predict(k_row.reshape(1,-1))))
        preds_matrix[b] = np.maximum(preds, 0)

    med = np.median(preds_matrix, axis=0)
    m   = metricas(y_te, med, preds_matrix, nome=f"QSVM_{cen}")
    RESULTADOS[cen] = {**m, "preds_matrix": preds_matrix, "mediana": med,
                        "y_test": y_te, "regimes_te": regimes_te,
                        "tempo_s": time.time() - t0}
    print(f"    R²={m['R2']:.4f} | WIS={m['WIS']:.2f} | {RESULTADOS[cen]['tempo_s']/60:.1f} min")

  QSVM REGRESSÃO COM ROTEAMENTO POR REGIME

  Cenário C1...
    Regime 2: poucos exemplos (0), usa SVR global
    R²=-0.0551 | WIS=2314.73 | 0.0 min

  Cenário C2...
    Regime 2: poucos exemplos (0), usa SVR global
    R²=-0.1429 | WIS=3421.82 | 0.0 min

  Cenário C3...
    R²=-16.7413 | WIS=404.41 | 0.0 min


In [ ]:
# ── Tabela de resultados ──────────────────────────────────────────────────
print(f"\n{'='*75}")
print(f"{'FASE 6 — QSVM com Quantum Kernel + Roteamento por Regime':^75}")
print(f"{'='*75}")
print(f"{'Cenário':<10} {'R²':>8} {'RMSE':>10} {'WIS':>10} {'Acc.Regime':>12}")
print("-" * 75)
for cen, r in RESULTADOS.items():
    acc = RESULTADOS_CLF[cen]["acc"]
    print(f"{cen:<10} {r['R2']:>8.4f} {r['RMSE']:>10.1f} {r['WIS']:>10.2f} {acc:>12.4f}")
print("=" * 75)

# ── Plot: predição + coloração por regime ─────────────────────────────────
CORES_REG = {0: "tab:blue", 1: "tab:orange", 2: "tab:red"}
LABEL_REG  = {0: "Declínio", 1: "Endemia", 2: "Crescimento"}

fig, axes = plt.subplots(3, 1, figsize=(14, 11))
for i, (cen, r) in enumerate(RESULTADOS.items()):
    ax  = axes[i]
    sem = np.arange(len(r["y_test"]))
    p10 = np.percentile(r["preds_matrix"], 10, axis=0)
    p90 = np.percentile(r["preds_matrix"], 90, axis=0)
    ax.fill_between(sem, p10, p90, alpha=0.15, color="gray", label="IC 80%")
    ax.plot(sem, r["y_test"], "k-", lw=1.5, label="casos_est (real)", zorder=5)
    ax.plot(sem, r["mediana"], "--", lw=1.5, color="purple",
            label=f"QSVM  R²={r['R2']:.3f}", zorder=4)
    # Colorir fundo por regime previsto
    for j, reg in enumerate(r["regimes_te"]):
        ax.axvspan(j - 0.5, j + 0.5, alpha=0.08, color=CORES_REG[reg])
    # Legenda de regime
    from matplotlib.patches import Patch
    patches = [Patch(color=CORES_REG[rr], alpha=0.3, label=LABEL_REG[rr]) for rr in [0,1,2]]
    handles, labels_ = ax.get_legend_handles_labels()
    ax.legend(handles=handles+patches, fontsize=8, loc="upper left")
    ax.set_title(f"{cen}: {CENARIOS[cen]['nome']}", fontweight="bold")
    ax.set_ylabel("casos_est"); ax.grid(alpha=0.3)
axes[-1].set_xlabel("Semana epidemiológica")
plt.suptitle("Fase 6 — QSVM: Predição + Regime Epidemiológico Detectado", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("fase3_qsvm_pred_regime.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Salva resultados ───────────────────────────────────────────────────────
import json as _json
resumo = {}
for cen, r in RESULTADOS.items():
    resumo[cen] = {k: round(float(v), 4) for k, v in r.items() if isinstance(v, float)}
    resumo[cen]["acc_regime"] = round(float(RESULTADOS_CLF[cen]["acc"]), 4)
with open("fase6_resultados.json", "w") as f:
    _json.dump(resumo, f, indent=2)
print("[SALVO] fase6_resultados.json")


         FASE 6 — QSVM com Quantum Kernel + Roteamento por Regime          
Cenário          R²       RMSE        WIS   Acc.Regime
---------------------------------------------------------------------------
C1          -0.0551     5878.5    2314.73       0.8601
C2          -0.1429     7416.3    3421.82       0.8242
C3         -16.7413      796.2     404.41       0.8889
[SALVO] fase6_resultados.json


In [18]:
# ── Figura previsto vs. observado (padrao dos demais modelos) ──
from utils_qml import plot_pred
plot_pred(RESULTADOS,
          "Fase 3 — QSVM: Predição vs. Observado (dados reais DF 2022-2025)",
          "fase3_qsvm_pred_vs_obs.png")


[SALVO] fase3_qsvm_pred_vs_obs.png


In [19]:
# ── Validação do pipeline de dados (integração) ──────────────────────────────
validar_pipeline(dataset, splits)


[PIPELINE OK] 13 features | 4 splits | serie=179 semanas | cenarios C1/C2/C3 prontos


True

In [20]:
import json as _json, os as _os

## Justificativa dos Hiperparâmetros - QSVM com Quantum Kernel

| Hiperparâmetro | Valor | Justificativa | Referência |
|---|---|---|---|
| `n_qubits` | 6 | 2⁶ = 64 dim. de Hilbert para o kernel; com 8 features de entrada, os 6 primeiros são mapeados (Rt e lags dominantes por importância RF) | Preskill (2018). Quantum, 2, 79 |
| `n_layers_kernel` | 2 | Havlíček et al. (2019) demonstram que 2 repetições do ZZ Feature Map são suficientes para criar separabilidade quântica não-clássica; mais repetições saturam o kernel | Havlíček et al. (2019). *Supervised learning with quantum-enhanced feature spaces*. Nature, 567, 209–212 |
| ZZ Feature Map | — | Feature map com entanglement ZZ codifica correlações de segunda ordem entre features epidemiológicas (Rt×p_rt1, casos×Rt) — relevante para dinâmica de surto | Havlíček et al. (2019). Nature |
| `C_svm` | 10.0 | Regularização SVM: C alto (menos regularização) é adequado para séries epidemiológicas onde o sinal é forte e o ruído é baixo (dados oficiais InfoDengue) | Cortes & Vapnik (1995). *Support-vector networks*. Machine Learning, 20(3) |
| `epsilon_svr` | 0.1 | Margem ε compatível com variabilidade natural de ~10% nas estimativas de casos_est do InfoDengue (método nowcasting) | — |
| Limiares de regime | Rt < 0.85 / > 1.20 | Calibrados na série histórica DF (2022–2025): Rt < 0.85 indica declínio sustentado; Rt > 1.20 indica crescimento epidêmico; zona de endemia 0.85–1.20 | Cori et al. (2013). Am. J. Epidemiol. |
| `N_BOOT` | 5 | Perturbação de 10% dos labels de regime para estimativa de incerteza; 5 réplicas — protocolo WIS | Bracher et al. (2021) |

> **Justificativa do roteamento por regime:** a dinâmica de dengue no DF apresenta três regimes distintos (declínio, endemia, crescimento) com dinâmicas não-lineares incompatíveis — um único SVR global viola a hipótese de estacionaridade.

In [21]:
CONFIG = CONFIG if "CONFIG" in dir() else {}
SCHEMA_INFO = {
    "algoritmo": "QSVM-QuantumKernel",
    "fase": 7,
    "tipo": "kernel",
    "n_parametros_quanticos": 0,
    "config": CONFIG,
}
# Adiciona acurácia de regime em cada cenário
for cen in RESULTADOS:
    RESULTADOS[cen]["acc_regime"] = float(RESULTADOS_CLF[cen]["acc"])
doc = salvar_padrao(RESULTADOS, SCHEMA_INFO)
validar_json_saida(doc, contexto="Fase6_QSVM_QuantumKernel")
validar_golden(doc, contexto="Fase6_QSVM_QuantumKernel")
# ── MLflow: registro automático do experimento ────────────────────────────────
_MLFLOW = False  # tracking desativado (entregável)
if _MLFLOW:
    with mlflow.start_run(run_name="Fase6_QSVM_QuantumKernel"):
        mlflow.log_params(SCHEMA_INFO.get("config", {}))
        mlflow.log_param("algoritmo",  SCHEMA_INFO.get("algoritmo", ""))
        mlflow.log_param("fase",       SCHEMA_INFO.get("fase", 0))
        mlflow.log_param("tipo",       SCHEMA_INFO.get("tipo", ""))
        for _cen in ["C1", "C2", "C3"]:
            if _cen in doc:
                mlflow.log_metric(f"WIS_{_cen}",      doc[_cen].get("WIS", float("nan")))
                mlflow.log_metric(f"WIS_norm_{_cen}", doc[_cen].get("WIS_norm", float("nan")))
                mlflow.log_metric(f"R2_{_cen}",       doc[_cen].get("R2", float("nan")))
                mlflow.log_metric(f"RMSE_{_cen}",     doc[_cen].get("RMSE", float("nan")))


[PADRAO] fase07_qsvm-quantumkernel_resultados.json
  Algoritmo : QSVM-QuantumKernel
  Tipo      : kernel
  Parametros quanticos: 0
  C1: R2=-0.0551 | WIS=2314.73 | 0.1s
  C2: R2=-0.1429 | WIS=3421.82 | 0.1s
  C3: R2=-16.7413 | WIS=404.41 | 0.0s
[CONTRATO OK] [Fase6_QSVM_QuantumKernel] JSON valido — todos os campos e invariantes corretos
[GOLDEN OK] [Fase6_QSVM_QuantumKernel] Resultados dentro da tolerancia vs. referencia.
